# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Procesamiento de Datos Masivos (Big Data)** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Lab 04** </center>
---
**Profesor**: Pablo Camarillo Ramirez

Guillermo Alejandro Romero Vázquez

In [66]:
from SparkUtils import SparkUtils

from pyspark.sql.functions import get_json_object

In [67]:
# create Spark session
MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Lab 04: Data Unions and Joins Pipeline"

spark = SparkUtils(MASTER_URL, APP_NAME)._spark

spark

In [ ]:
# agencies dataset
columns_agencies = [
    ("agency_id", "int"),
    ("agency_info", "string")
]

schema_agencies = SparkUtils\
    .generate_schema(columns_agencies)

# reading JSON files
df_agencies = spark.read\
    .option("header", "true")\
    .schema(schema_agencies)\
    .csv("/opt/spark/work-dir/data/car_service/agencies/")

df_agencies = df_agencies\
    .withColumn("agency_name", get_json_object("agency_info", "$.agency_name"))\
    .withColumn("city", get_json_object("agency_info", "$.city"))\
    .drop("agency_info")

# df_agencies.show(truncate=False)

In [69]:
# brands dataset
columns_brands = [
    ("brand_id", "int"),
    ("brand_info", "string")
]

schema_brands = SparkUtils\
    .generate_schema(columns_brands)

# reading JSON files
df_brands = spark.read\
    .option("header", "true")\
    .schema(schema_brands)\
    .csv("/opt/spark/work-dir/data/car_service/brands/")

df_brands = df_brands\
    .withColumn("brand_name", get_json_object("brand_info", "$.brand_name"))\
    .withColumn("country", get_json_object("brand_info", "$.country"))\
    .drop("brand_info")

# df_brands.show(truncate=False)

In [70]:
# cars dataset
columns_cars = [
    ("car_id", "int"),
    ("car_info", "string")
]

schema_cars = SparkUtils\
    .generate_schema(columns_cars)

df_cars = spark.read\
    .option("header", "true")\
    .schema(schema_cars)\
    .csv("/opt/spark/work-dir/data/car_service/cars/")

df_cars = df_cars\
    .withColumn("brand_id", get_json_object("car_info", "$.brand_id"))\
    .withColumn("car_name", get_json_object("car_info", "$.car_name"))\
    .withColumn("price_per_day", get_json_object("car_info", "$.price_per_day"))\
    .drop("car_info")

# df_cars.show(5, truncate=False)

In [71]:
# customers dataset
columns_customers = [
    ("customer_id", "int"),
    ("customer_info", "string")
]

schema_customers = SparkUtils\
    .generate_schema(columns_customers)

df_customers = spark.read\
    .option("header", "true")\
    .schema(schema_customers)\
    .csv("/opt/spark/work-dir/data/car_service/customers/")

df_customers = df_customers\
    .withColumn("customer_name", get_json_object("customer_info", "$.customer_name"))\
    .withColumn("city", get_json_object("customer_info", "$.city"))\
    .withColumn("age", get_json_object("customer_info", "$.age"))\
    .drop("customer_info")

# df_customers.show(5, truncate=False)

In [72]:
# rentals dataset
columns_rentals = [
    ("rental_id", "int"),
    ("rental_info", "string")
]

schema_rentals = SparkUtils\
    .generate_schema(columns_rentals)

df_rentals = spark.read\
    .option("header", "true")\
    .schema(schema_rentals)\
    .csv("/opt/spark/work-dir/data/car_service/rentals/")

df_rentals = df_rentals\
    .withColumn("car_id", get_json_object("rental_info", "$.car_id"))\
    .withColumn("customer_id", get_json_object("rental_info", "$.customer_id"))\
    .withColumn("agency_id", get_json_object("rental_info", "$.agency_id"))\
    .drop("rental_info")

# df_rentals.show(5, truncate=False)

In [73]:
df_result = df_rentals\
    .join(df_cars, on="car_id", how="left")\
    .join(df_agencies, on="agency_id", how="left")\
    .join(df_customers, on="customer_id", how="left")\
    .select("rental_id", "car_name", "agency_name", "customer_name")

df_result.show(5, truncate=False)

+---------+-----------------------+-------------+---------------+
|rental_id|car_name               |agency_name  |customer_name  |
+---------+-----------------------+-------------+---------------+
|11891    |Wallace-Carlson Model 9|NYC Rentals  |Margaret Jones |
|11892    |Grimes-Green Model 8   |LA Car Rental|Albert Williams|
|11893    |Stewart-Allen Model 5  |SF Cars      |Caleb Fleming  |
|11894    |Campos PLC Model 4     |NYC Rentals  |Andrew Butler  |
|11895    |Wagner LLC Model 1     |SF Cars      |Kristin Potts  |
+---------+-----------------------+-------------+---------------+
only showing top 5 rows


In [74]:
spark.stop()